# DSA4212 small LLM
Set Runtime > Change runtime type > T4 GPU before running.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/dsa4212/data /content/drive/MyDrive/dsa4212/runs

In [ ]:
!git clone https://github.com/phoebeyap04/dsa4212-small-llm.git /content/repo
%cd /content/repo
!pip install tiktoken datasets tokenizers

## Data
Only run the first cell the very first time (takes 20-40 min). It saves the data to Drive. After that, just run the second cell.

In [ ]:
# FIRST TIME ONLY
%cd /content/repo/data/tinystories
!python prepare.py
!cp train_small.bin train_large.bin val.bin test.bin /content/drive/MyDrive/dsa4212/data/
%cd /content/repo

In [ ]:
# every other time: copy the data from Drive
!cp /content/drive/MyDrive/dsa4212/data/*.bin /content/repo/data/tinystories/
!ls -lh /content/repo/data/tinystories

## Settings
Change these for each run. For a quick test use budget = 300.0 (keep the .0).

In [ ]:
size = 'small'    # small, medium or large
pool = 'small'    # small or large
seed = 1337       # 1337 or 2024
budget = 3600.0

run_name = f'{size}_{pool}_seed{seed}'
if budget != 3600.0:
    run_name = run_name + '_test'
out = f'/content/drive/MyDrive/dsa4212/runs/{run_name}'
print(out)

## Train

In [ ]:
!python train.py config/ts_{size}.py --train_file=train_{pool}.bin --seed={seed} --time_budget={budget} --out_dir={out}

If Colab disconnects in the middle of a run, rerun the cells above (except training), then run this to continue from the last checkpoint:

In [ ]:
!python train.py config/ts_{size}.py --train_file=train_{pool}.bin --seed={seed} --time_budget={budget} --out_dir={out} --init_from=resume

## Results

In [ ]:
!cat {out}/summary.json
!python sample.py --out_dir={out} --num_samples=2 --max_new_tokens=200